# reglscatterpy — examples cookbook

One short, self-contained example per feature (real **pbmc3k**, 2,638 PBMCs).
Every cell says what it does in a comment — skim, run, copy. For the narrated
walkthrough see [`reglscatterpy_tour.ipynb`](reglscatterpy_tour.ipynb).

Run the setup cell first; then any cell stands alone.

In [1]:
import numpy as np, pandas as pd, scanpy as sc
import reglscatterpy as rs
adata = sc.read_h5ad("data/pbmc3k_processed.h5ad")
adata                                   # 2638 cells x 1838 genes, obs: louvain / n_genes / percent_mito

/home/goguxor/miniconda3/envs/reglpy/lib/python3.11/site-packages/anndata/_io/h5ad.py:267: FutureWarning: Moving element from .uns['neighbors']['distances'] to .obsp['distances'].

This is where adjacency matrices should go now.
  return AnnData(**{
/home/goguxor/miniconda3/envs/reglpy/lib/python3.11/site-packages/anndata/_io/h5ad.py:267: FutureWarning: Moving element from .uns['neighbors']['connectivities'] to .obsp['connectivities'].

This is where adjacency matrices should go now.
  return AnnData(**{


AnnData object with n_obs × n_vars = 2638 × 1838
    obs: 'n_genes', 'percent_mito', 'n_counts', 'louvain'
    var: 'n_cells'
    uns: 'draw_graph', 'louvain', 'louvain_colors', 'neighbors', 'pca', 'rank_genes_groups'
    obsm: 'X_pca', 'X_tsne', 'X_umap', 'X_draw_graph_fr'
    varm: 'PCs'
    obsp: 'distances', 'connectivities'

## Basics

### Colour by a cell-type column

In [2]:
rs.scatterplot(adata, basis="umap", color_by="louvain")   # categorical -> legend with counts

<reglscatterpy._widget._make_classes.<locals>.ReglScatter object at 0x7fe1dd2d7b90>

### Colour by a gene

In [3]:
rs.scatterplot(adata, basis="umap", color_by="CST3")      # gene name -> continuous colour bar

<reglscatterpy._widget._make_classes.<locals>.ReglScatter object at 0x7fe1dd2f3450>

### A plain DataFrame (give the x / y columns)

In [5]:
df = pd.DataFrame({"x": adata.obsm["X_umap"][:,0], "y": adata.obsm["X_umap"][:,1],
                   "cluster": adata.obs["louvain"].values})
rs.scatterplot(df, x="x", y="y", color_by="cluster")

<reglscatterpy._widget._make_classes.<locals>.ReglScatter object at 0x7fe7dbac7810>

### Size / width

In [6]:
rs.scatterplot(adata, basis="umap", color_by="louvain", width=500)   # px; width=None fills the cell

<reglscatterpy._widget._make_classes.<locals>.ReglScatter object at 0x7fe7dbacd210>

## Colour & encoding

### Continuous palette + fixed range + diverging centre

In [4]:
rs.scatterplot(adata, basis="umap", color_by="CST3",
               cmap="magma", vmin=0, vmax=4)          # or center_zero=True for a diverging map

<reglscatterpy._widget._make_classes.<locals>.ReglScatter object at 0x7fe1dd2fbbd0>

### Encode a 2nd variable on point size or opacity (obs **or** gene)

In [5]:
rs.scatterplot(adata, basis="umap", color_by="louvain", size_by="n_genes", size=20)   # or opacity_by="CST3"

<reglscatterpy._widget._make_classes.<locals>.ReglScatter object at 0x7fe1dd2f8190>

### Richer hover tooltips

In [6]:
rs.scatterplot(adata, basis="umap", color_by="louvain",
               tooltip_by=["n_genes", "percent_mito", "CST3"])   # obs cols and/or genes

<reglscatterpy._widget._make_classes.<locals>.ReglScatter object at 0x7fe1dd471310>

### Scanpy-style outline on every point

In [7]:
rs.scatterplot(adata, basis="umap", color_by="louvain", add_outline=True, size=10, outline_width=(0.2, 0))   # add_outline=True for categorical color_by

<reglscatterpy._widget._make_classes.<locals>.ReglScatter object at 0x7fe1dd4728d0>

### Theme (light / dark / auto)

In [8]:
rs.scatterplot(adata, basis="umap", color_by="louvain", theme="dark")   # or rs.set_theme("auto") once"

<reglscatterpy._widget._make_classes.<locals>.ReglScatter object at 0x7fe1dd332250>

## Lasso → ask questions (the round-trip)

These need the live widget (`interactive=True`). Here we drive the selection
from Python so the cookbook runs without a mouse — **in the UI you'd just lasso.**

In [27]:
w = rs.scatterplot(adata, basis="umap", color_by="louvain", interactive=True)
# pretend we lassoed the B cells:
w.selection = adata.obs_names[adata.obs["louvain"] == "B cells"]
print("selected:", len(w.selection), "cells")
w                                       # the widget (lasso here instead, if you like)

selected: 342 cells


<reglscatterpy._widget._make_classes.<locals>.ReglScatter object at 0x7fe7de401fd0>

### Read the selection back

In [28]:
adata[w.selection]                      # subset the AnnData  (== w.subset())

View of AnnData object with n_obs × n_vars = 509 × 1838
    obs: 'n_genes', 'percent_mito', 'n_counts', 'louvain'
    var: 'n_cells'
    uns: 'draw_graph', 'louvain', 'louvain_colors', 'neighbors', 'pca', 'rank_genes_groups'
    obsm: 'X_pca', 'X_tsne', 'X_umap', 'X_draw_graph_fr'
    varm: 'PCs'
    obsp: 'distances', 'connectivities'

### What is the selection made of?

In [29]:
w.composition("louvain")                # count + fraction per category

,count,fraction
louvain,,
CD8 T cells,304,0.597250
CD4 T cells,104,0.204322
NK cells,100,0.196464
Megakaryocytes,1,0.001965
B cells,0,0.000000
CD14+ Monocytes,0,0.000000
FCGR3A+ Monocytes,0,0.000000
Dendritic cells,0,0.000000


### Label the lassoed cells straight into obs

In [30]:
w.annotate("cell_type", "B cells")      # writes adata.obs['cell_type'] for those cells
adata.obs["cell_type"].value_counts(dropna=False)

cell_type
NaN        2129
B cells     509
Name: count, dtype: int64

### Top marker genes of the selection (scanpy-native result)

In [31]:
w.diff_expression(n=10)                          # saved to adata.uns['rank_genes_groups']
sc.get.rank_genes_groups_df(adata, group="A").head(10)

,names,scores,logfoldchanges,pvals,pvals_adj
0,NKG7,26.078094,4.542823,6.461709e-150,1.187662e-146
1,CCL5,25.542385,4.202890,6.671738e-144,6.131327e-141
2,GZMA,22.145485,4.073375,1.153131e-108,6.000081e-106
3,CTSW,22.139881,3.121782,1.305785e-108,6.000081e-106
4,CST7,22.069340,4.090913,6.229534e-108,2.289977e-105
5,IL32,18.875452,1.860713,1.815640e-79,5.561912e-77
6,PTPRCAP,15.763508,1.309948,5.546653e-56,1.456393e-53
7,PRF1,14.699117,3.191039,6.530597e-49,1.500405e-46
8,LYAR,13.376814,2.633319,8.261378e-41,1.687157e-38
9,GZMK,13.261542,4.297566,3.868584e-40,7.110457e-38


### Same DE, forced onto the **GPU** (cupy) — or require scanpy

In [32]:
w.diff_expression(n=10, engine="gpu")           # GPU Welch t-test; engine="scanpy" to require scanpy
# engine="auto" (default) uses scanpy and warns if it must fall back

/home/goguxor/Desktop/reglscatterpy/src/reglscatterpy/_widget.py:547: RuntimeWarning: invalid value encountered in log2
  lfc = np.log2((ma + 1e-9) / (mb + 1e-9))


{'params': {'groupby': 'group',
  'reference': 'rest',
  'method': 't-test',
  'use_raw': False,
  'layer': None,
  'corr_method': 'none'},
 'names': rec.array([('NKG7',), ('CCL5',), ('GZMA',), ('CST7',), ('CTSW',),
            ('IL32',), ('S100A8',), ('AIF1',), ('LGALS2',), ('FCN1',)],
           dtype=[('A', 'O')]),
 'scores': rec.array([( 34.06887255,), ( 33.70158865,), ( 26.89801501,),
            ( 26.64587064,), ( 26.55539253,), ( 24.05931553,),
            (-22.52877595,), (-22.06427912,), (-21.93361406,),
            (-19.74351775,)],
           dtype=[('A', '<f8')]),
 'logfoldchanges': rec.array([(nan,), (nan,), (nan,), (nan,), (nan,), (nan,), (nan,), (nan,),
            (nan,), (nan,)],
           dtype=[('A', '<f8')]),
 'pvals': rec.array([(3.17977381e-142,), (2.45867596e-142,), (2.04158426e-103,),
            (6.61329361e-102,), (5.39417018e-104,), (5.54048477e-098,),
            (1.11808328e-102,), (6.34665817e-095,), (4.71022326e-098,),
            (2.32258887e-079,)],
  

### DE **between metadata levels** inside the lasso

In [33]:
# split the selection by an obs column and compare its levels (one col per level):
adata.obs["batch"] = pd.Categorical(np.where(adata.obs["n_genes"] > 800, "hi", "lo"))
w.selection = list(range(adata.n_obs))          # (whole set, for the demo)
res = w.diff_expression_by("batch")
res["names"].dtype.names                         # ('hi', 'lo')

('hi', 'lo')

### Persistently highlight a chosen subset (survives new lassoes)

In [36]:
hi = np.where(adata.obs["louvain"] == "NK cells")[0]
w.highlight(hi, color="red")                    # ring + size bump; w.highlight([]) clears

<reglscatterpy._widget._make_classes.<locals>.ReglScatter object at 0x7fe7de401fd0>

## Linked / multi-panel views

### One embedding, several colours → linked grid

In [9]:
rs.scatterplot(adata, basis="umap", color_by=["louvain", "CST3", "NKG7"], ncols=3)

_ComposedPlots(children=(<reglscatterpy._widget._make_classes.<locals>.ReglScatter object at 0x7fe1dd3328d0>, …

### Several embeddings, one colour → linked grid

In [38]:
rs.scatterplot(adata, basis=["umap", "pca"], color_by="louvain")

_ComposedPlots(children=(<reglscatterpy._widget._make_classes.<locals>.ReglScatter object at 0x7fe7c09a4e10>, …

### Compose pre-built plots (linked camera + selection)

In [ ]:
from reglscatterpy import scatterplot, compose
a = scatterplot(adata, basis="umap", color_by="louvain")
b = scatterplot(adata, basis="pca",  color_by="louvain")
compose([a, b])

## Big data

In [ ]:
# density-preserving cap (keeps rare types) — default is "auto" (500k):
rs.scatterplot(adata, basis="umap", color_by="louvain", max_points=1000)   # try a tiny cap to see the caption
# max_points=None = every point on the GPU; progressive=True = detail-on-zoom for >4M

## Export a standalone, offline HTML (no kernel)

In [10]:
import tempfile, os
w2 = rs.scatterplot(adata, basis="umap", color_by="louvain", show=False)
out = os.path.join(tempfile.mkdtemp(), "umap.html")
rs.save_html(w2, out)                    # self-contained; opens in any browser
print(out, os.path.getsize(out)//1024, "KB")

/tmp/tmp90sfnhn3/umap.html 578 KB


## Spatial + morph (public squidpy dataset)

Anything with `obsm['spatial']` plots with `basis="spatial"`. This uses squidpy's
public **mouse-brain Visium** — `pip install squidpy` (downloads ~a few MB once).

In [11]:
import squidpy as sq
sp = sq.datasets.visium_hne_adata()      # 2,688 spots; has spatial + X_umap + clusters

INFO     Downloading visium_hne_adata.h5ad from https://exampledata.scverse.org/squidpy/visium_hne_adata.h5ad      


/home/goguxor/miniconda3/envs/reglpy/lib/python3.11/site-packages/docrep/decorators.py:43: SyntaxWarning: 'n_jobs' is not a valid key!
  doc = func(self, args[0].__doc__, *args[1:], **kwargs)
/home/goguxor/miniconda3/envs/reglpy/lib/python3.11/site-packages/docrep/decorators.py:43: SyntaxWarning: 'show_progress_bar' is not a valid key!
  doc = func(self, args[0].__doc__, *args[1:], **kwargs)


  0%|                                               | 0.00/329M [00:00<?, ?B/s]

### Tissue map — colour by cluster

In [12]:
rs.scatterplot(sp, basis="spatial", color_by="cluster")

<reglscatterpy._widget._make_classes.<locals>.ReglScatter object at 0x7fe1979dca10>

### A gene, in situ

In [13]:
rs.scatterplot(sp, basis="spatial", color_by="Olfm1")     # any var_name; e.g. Plp1, Mbp, Ttr

<reglscatterpy._widget._make_classes.<locals>.ReglScatter object at 0x7fe191874250>

### Morph UMAP ↔ tissue

In [14]:
w = rs.scatterplot(sp, basis="umap", color_by="cluster", interactive=True)
w    # display it, then run the next cell

<reglscatterpy._widget._make_classes.<locals>.ReglScatter object at 0x7fe1913af1d0>

In [ ]:
w.morph_to("spatial", duration=2000)     # animate UMAP -> tissue coordinates
# w.morph_to("umap")                       # ...and back